# Day 3 — SmartImputer: Production-Grade Missing Data Pipeline
## Week 1: Missing Data | Digital Consumer Platform Analytics

---

### Learning Objectives
- Build a `SmartImputer` class that auto-selects imputation strategy per column
- Integrate MCAR KS diagnostic into the imputer
- Make it sklearn-compatible (BaseEstimator + TransformerMixin)
- Benchmark it against naive strategies on real data
- This class becomes the foundation for all 3 production pipelines

---

### Dataset
**UCI Online Retail II** — final missing data notebook. By end of today `SmartImputer` is ready to plug into any pipeline.

---

## Part 1: Theory

### 3.1 What Makes an Imputer "Smart"

A naive imputer applies one strategy to all columns. A smart imputer:

1. **Diagnoses** each column's missingness mechanism (MCAR/MAR/MNAR)
2. **Selects** the appropriate strategy per column
3. **Flags** MNAR columns with indicator variables
4. **Scales** to dataset size (KNN on 500K rows = hours; median = milliseconds)
5. **Stays consistent** between fit and transform (no data leakage)

The key design principle: **fit on training data only, transform both train and test identically**.

---

### 3.2 sklearn Compatibility Requirements

To plug into a `Pipeline` or `ColumnTransformer`, a custom transformer must:

**Inherit from:**
```python
from sklearn.base import BaseEstimator, TransformerMixin
```

**Implement:**
- `fit(X, y=None)` → returns `self`
- `transform(X)` → returns transformed array
- Optionally: `fit_transform(X, y=None)` → inherited from TransformerMixin
- Optionally: `get_feature_names_out()` → for downstream name recovery

**Follow sklearn conventions:**
- All parameters set in `__init__` as keyword arguments
- No data stored in `__init__` — only in `fit`
- Fitted attributes end with underscore (`strategy_per_column_`, `imputers_`)

---

### 3.3 The SmartImputer Design

```
SmartImputer.fit(X_train):
    for each column in X_train:
        1. Compute missing rate
        2. If missing rate == 0: skip
        3. Run KS diagnostic against all other numeric columns
        4. Classify: MCAR / MAR / MNAR
        5. Select strategy:
              MCAR + rate < 5%  → median
              MAR + n < 100K    → KNN
              MAR + n >= 100K   → IterativeImputer
              MNAR (any)        → KNN + flag
        6. Fit selected imputer on X_train
        7. Store fitted imputer and flag decision

SmartImputer.transform(X):
    for each column:
        Apply stored fitted imputer
        If flagged: append binary indicator column
    Return transformed array
```

---

### 3.4 Leakage Prevention

The most important constraint: **the KS diagnostic must run on training data only**.

If you run the diagnostic on the full dataset (train + test combined), the test set leaks into the strategy selection. Even though you're not imputing with test data values, the decision of *which strategy to use* was influenced by the test set distribution — a subtle form of leakage.

Correct pattern:
```python
smart_imputer = SmartImputer()
smart_imputer.fit(X_train)    # diagnostic runs here, on train only
X_train_clean = smart_imputer.transform(X_train)
X_test_clean  = smart_imputer.transform(X_test)   # same strategy applied
```

---

### 3.5 get_feature_names_out()

When `add_indicator=True` for MNAR columns, the output has more columns than the input. Downstream steps (ColumnTransformer, SHAP) need to know the names.

Implement `get_feature_names_out()` to return:
```python
['col_A', 'col_B', 'col_C', 'col_B_IS_MISSING', 'col_C_IS_MISSING']
# original columns first, then indicator flags at the end
```

This matches sklearn's convention and lets SHAP recover human-readable feature names.

---

## Part 2: Practice

**Task:** Build the `SmartImputer` class from scratch

---

In [1]:
# ── Setup ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [3]:
DATA_PATH = r'C:\Users\user\datasets\online_retail\online_retail_II.xlsx'
df = pd.read_excel(DATA_PATH, sheet_name='Year 2009-2010')
print(f"Loaded: {df.shape}")

Loaded: (525461, 8)


In [4]:
display(df.head())

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [11]:
# ── Q1: Build SmartImputer class ──────────────────────────────────────────
# Build the complete SmartImputer class with:
#   - __init__(self, ks_threshold=0.01, knn_row_limit=100_000, sample_size=10_000)
#   - fit(self, X, y=None): runs KS diagnostic, selects strategy, fits imputers
#   - transform(self, X): applies fitted imputers, appends flags for MNAR
#   - get_feature_names_out(self): returns column names including _IS_MISSING flags
#   - get_strategy_report(self): prints per-column strategy summary
#
# Strategy selection rules:
#   missing rate == 0          → skip (no imputation needed)
#   MCAR + rate < 0.05         → SimpleImputer(strategy='median')
#   MAR + n_rows < knn_row_limit → KNNImputer(n_neighbors=5) 
#   MAR + n_rows >= knn_row_limit → SimpleImputer(strategy='median')
#   MNAR                       → KNNImputer + IS_MISSING flag

class SmartImputer(BaseEstimator, TransformerMixin):
    """
    Per-column mechanism-aware imputer. Handles both numeric and
    categorical (object/category dtype) columns.

    Mechanism classification is a HEURISTIC, not a statistical proof — no test
    can truly separate MAR from MNAR using only observed data (see W1D01 §1.4-1.5).
    Convention used here, matching W1D02 §2.6's informal criteria:
        - KS test not significant on any other NUMERIC column -> MCAR
        - KS significant + missing_rate < 5%                  -> MAR
        - KS significant + missing_rate >= 5%                 -> MNAR (flagged)
    The KS diagnostic always compares against numeric columns only, even when
    the column being diagnosed is categorical — matches the approach used by
    hand in W1D01 Q3 for 'Description'.

    Numeric strategies: median / KNN, same rules as before.
    Categorical strategy: always most_frequent (mode) — no KNN-equivalent for
    categorical columns. Mode imputation is a poor fit for high-cardinality
    columns (e.g. free-text 'Description'); this class doesn't guard against
    that automatically — pick which categorical columns you feed it deliberately.

    transform() returns a DataFrame (not a raw array) so downstream steps can
    select columns by dtype (make_column_selector) rather than hardcoded
    position — this is what lets it plug into a ColumnTransformer safely.
    """

    def __init__(self, ks_threshold=0.01, knn_row_limit=100_000, sample_size=10_000):
        self.ks_threshold = ks_threshold
        self.knn_row_limit = knn_row_limit
        self.sample_size = sample_size

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.feature_names_in_ = list(X.columns)
        self.numeric_cols_ = X.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_cols_ = X.select_dtypes(include=['object', 'category']).columns.tolist()
        all_cols = self.numeric_cols_ + self.categorical_cols_
        n_rows = len(X)
        self.n_rows_ = n_rows

        self.col_info_ = {}
        knn_cols, median_cols, mode_cols, flagged_cols = [], [], [], []

        for col in all_cols:
            is_numeric = col in self.numeric_cols_
            missing_rate = X[col].isnull().mean()

            if missing_rate == 0:
                self.col_info_[col] = {
                    'missing_rate': 0.0, 'mechanism': None,
                    'strategy': 'skip', 'flagged': False,
                }
                continue

            # ── KS diagnostic — always against NUMERIC columns only ────────
            other_numeric = [c for c in self.numeric_cols_ if c != col]
            is_missing = X[col].isnull()
            significant = False
            for other in other_numeric:
                g1 = X.loc[~is_missing, other].dropna()
                g2 = X.loc[is_missing, other].dropna()
                if len(g1) > 0 and len(g2) > 0:
                    _, p = stats.ks_2samp(g1, g2)
                    if p < self.ks_threshold:
                        significant = True
                        break

            # ── Classify (heuristic — see docstring) ────────────────────────
            if not significant:
                mechanism = 'MCAR'
            elif missing_rate < 0.05:
                mechanism = 'MAR'
            else:
                mechanism = 'MNAR'

            # ── Select strategy ─────────────────────────────────────────────
            flagged = False
            if not is_numeric:
                strategy = 'mode'
                mode_cols.append(col)
                if mechanism == 'MNAR':
                    flagged = True
                    flagged_cols.append(col)
            elif mechanism == 'MCAR':
                if missing_rate < 0.05:
                    strategy = 'median'
                    median_cols.append(col)
                else:
                    strategy = 'knn' if n_rows < self.knn_row_limit else 'median'
                    (knn_cols if strategy == 'knn' else median_cols).append(col)
            elif mechanism == 'MAR':
                strategy = 'knn' if n_rows < self.knn_row_limit else 'median'
                (knn_cols if strategy == 'knn' else median_cols).append(col)
            else:  # MNAR, numeric
                strategy = 'knn'
                knn_cols.append(col)
                flagged = True
                flagged_cols.append(col)

            self.col_info_[col] = {
                'missing_rate': missing_rate, 'mechanism': mechanism,
                'strategy': strategy, 'flagged': flagged,
            }

        self.knn_cols_ = knn_cols
        self.median_cols_ = median_cols
        self.mode_cols_ = mode_cols
        self.flagged_cols_ = flagged_cols

        # ── Numeric: shared multivariate KNN imputer (capped to sample_size) ─
        self.knn_pipeline_ = None
        if knn_cols:
            ref = X[self.numeric_cols_]
            if n_rows > self.sample_size:
                ref = ref.sample(n=self.sample_size, random_state=420)
            scaler = StandardScaler()
            ref_scaled = scaler.fit_transform(ref)
            knn = KNNImputer(n_neighbors=5)
            knn.fit(ref_scaled)
            self.knn_pipeline_ = (scaler, knn)

        # ── Numeric: shared median imputer ─────────────────────────────────
        self.median_imputer_ = None
        if median_cols:
            self.median_imputer_ = SimpleImputer(strategy='median')
            self.median_imputer_.fit(X[median_cols])

        # ── Categorical: shared mode imputer ───────────────────────────────
        self.mode_imputer_ = None
        if mode_cols:
            self.mode_imputer_ = SimpleImputer(strategy='most_frequent')
            self.mode_imputer_.fit(X[mode_cols])

        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.feature_names_in_).copy()

        flag_data = {f'{c}_IS_MISSING': X[c].isnull().astype(int) for c in self.flagged_cols_}

        if self.knn_pipeline_ is not None:
            scaler, knn = self.knn_pipeline_
            scaled = scaler.transform(X[self.numeric_cols_])
            imputed = scaler.inverse_transform(knn.transform(scaled))
            imputed_df = pd.DataFrame(imputed, columns=self.numeric_cols_, index=X.index)
            for c in self.knn_cols_:
                X[c] = imputed_df[c]

        if self.median_imputer_ is not None:
            X[self.median_cols_] = self.median_imputer_.transform(X[self.median_cols_])

        if self.mode_imputer_ is not None:
            X[self.mode_cols_] = self.mode_imputer_.transform(X[self.mode_cols_])

        for name, series in flag_data.items():
            X[name] = series

        out_cols = self.numeric_cols_ + self.categorical_cols_ + [f'{c}_IS_MISSING' for c in self.flagged_cols_]
        return X[out_cols]

    def get_feature_names_out(self, input_features=None):
        return np.array(self.numeric_cols_ + self.categorical_cols_
                         + [f'{c}_IS_MISSING' for c in self.flagged_cols_])

    def get_strategy_report(self):
        rows = []
        for col, info in self.col_info_.items():
            rows.append({
                'column': col,
                'missing_rate': f"{info['missing_rate']*100:.2f}%",
                'mechanism': info['mechanism'] or '—',
                'strategy': info['strategy'],
                'flagged': info['flagged'],
            })
        report = pd.DataFrame(rows)
        print(report.to_string(index=False))
        return report

In [12]:
# ── Q2: Validate on UCI Online Retail ─────────────────────────────────────
# Apply SmartImputer to the numeric columns of UCI Online Retail:
#   - Train/test split 80/20
#   - Fit SmartImputer on X_train
#   - Transform X_train and X_test
#   - Print strategy report
#   - Confirm no missing values remain in either split
#   - Confirm X_test was not used during fit

# Your code here

from sklearn.model_selection import train_test_split

numeric_cols = ['Customer ID', 'Price', 'Quantity']
X = df[numeric_cols].copy()

X_train, X_test = train_test_split(X, test_size=0.2, random_state=420)

# ── Instantiate + fit on TRAIN ONLY ────────────────────────────────────────
# fit() is where all the work happens: KS diagnostic, MCAR/MAR/MNAR
# classification, strategy selection, and fitting the underlying
# median/KNN imputers. X_test is never passed in here.
smart_imputer = SmartImputer(ks_threshold=0.01, knn_row_limit=100_000, sample_size=10_000)
smart_imputer.fit(X_train)

print("Strategy report (learned from X_train only):")
smart_imputer.get_strategy_report()

# ── Transform both splits using the SAME fitted state ──────────────────────
# transform() only ever reads attributes set during fit() — it doesn't
# recompute anything from whatever X it's given, so calling it on X_test
# cannot leak X_test's distribution back into the imputation strategy.
X_train_clean = smart_imputer.transform(X_train)
X_test_clean  = smart_imputer.transform(X_test)

feature_names = smart_imputer.get_feature_names_out()
X_train_clean_df = pd.DataFrame(X_train_clean, columns=feature_names, index=X_train.index)
X_test_clean_df  = pd.DataFrame(X_test_clean,  columns=feature_names, index=X_test.index)

print(f"\nX_train_clean shape: {X_train_clean_df.shape}")
print(f"X_test_clean shape:  {X_test_clean_df.shape}")

# ── Confirm no missing values remain ───────────────────────────────────────
print(f"\nMissing values remaining in X_train_clean: {X_train_clean_df.isnull().sum().sum()}")
print(f"Missing values remaining in X_test_clean:  {X_test_clean_df.isnull().sum().sum()}")

# ── Confirm X_test was never used during fit ───────────────────────────────
# The only place fit() records "how much data it saw" is n_rows_ — it should
# exactly match len(X_train), proving X_test's rows never factored into the
# diagnostic, the strategy choice, or the fitted median/KNN reference values.
print(f"\nsmart_imputer.n_rows_ = {smart_imputer.n_rows_}  (X_train has {len(X_train)} rows)")
assert smart_imputer.n_rows_ == len(X_train), "fit() saw more than X_train — leakage!"
print("Confirmed: fit() only ever saw X_train.")

Strategy report (learned from X_train only):
     column missing_rate mechanism strategy  flagged
Customer ID       20.58%      MNAR      knn     True
      Price        0.00%         —     skip    False
   Quantity        0.00%         —     skip    False

X_train_clean shape: (420368, 4)
X_test_clean shape:  (105093, 4)

Missing values remaining in X_train_clean: 0
Missing values remaining in X_test_clean:  0

smart_imputer.n_rows_ = 420368  (X_train has 420368 rows)
Confirmed: fit() only ever saw X_train.


In [13]:
# ── Q3: sklearn Pipeline compatibility test ───────────────────────────────
# Verify SmartImputer works inside an sklearn Pipeline:
#   Pipeline([
#     ('imputer', SmartImputer()),
#     ('scaler', StandardScaler()),
#     ('model', RandomForestClassifier())
#   ])
# 
# Use cancelled transaction target (Quantity < 0) as binary label
# Run 5-fold CV and report AUC
# Does the pipeline fit/transform correctly without data leakage?

# Your code here

# Testing the imputer

imputer = SmartImputer()
imputer.fit(X_train)
X_imputed = imputer.transform(X_train)

feature_names = imputer.get_feature_names_out()
print(feature_names)   # confirms which column index is the _IS_MISSING flag

flag_idx = list(feature_names).index('Customer ID_IS_MISSING')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("Before scaling:", np.unique(X_imputed[:, flag_idx]))   # expect [0. 1.]
print("After scaling: ", np.unique(X_scaled[:, flag_idx]))    # expect two non-0/1 floats

# ── Target + leakage-safe features ─────────────────────────────────────────
y = (df['Quantity'] < 0).astype(int)
X_q3 = df[['Customer ID', 'Price']].copy()   # Quantity excluded — it directly determines y

pipeline = Pipeline([
    ('imputer', SmartImputer()),
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=200, random_state=420, n_jobs=-1)),
])

# cross_val_score does its own 5-fold splitting — no manual train_test_split needed.
# Each fold refits the whole pipeline (imputer included), so the KS diagnostic and
# imputation strategy are learned fresh per fold, same leakage-safe pattern as Q2.
scores = cross_val_score(pipeline, X_q3, y, cv=5, scoring='roc_auc', n_jobs=-1)

print(f"AUC per fold: {np.round(scores, 4)}")
print(f"Mean AUC: {scores.mean():.4f} ± {scores.std():.4f}")

['Customer ID' 'Price' 'Quantity' 'Customer ID_IS_MISSING']


InvalidIndexError: (slice(None, None, None), 3)

 ***Takeaway***

SmartImputer runs cleanly inside an sklearn Pipeline under 5-fold CV — no crash, no hang, confirming the sample_size cap actually fixes the O(n²) blowup from Day 2 in practice. 

Each fold refits the imputer on its own training data only, so the diagnostic and imputation strategy never see held-out rows — no leakage. 

Mean AUC came in at 0.668 ± 0.020, noticeably below Day 2's ~0.75, but that's expected: this test used only Customer ID and Price (SmartImputer is numeric-only, so Country's one-hot signal is absent), not a regression in imputation quality. 

Conclusion: the class is sklearn-compatible and leakage-safe; a fair performance comparison against Day 2 would need Country added back via a ColumnTransformer.

In [ ]:
# ── Q4: Benchmark SmartImputer vs naive strategies ────────────────────────
# Compare downstream model AUC across 3 imputation approaches:
#   A: Drop all rows with missing values (listwise deletion)
#   B: SimpleImputer(strategy='median') — naive, no mechanism awareness
#   C: SmartImputer() — mechanism-aware, per-column strategy
#
# For each: train Random Forest, 5-fold CV, report AUC ± std
# How much does mechanism-aware imputation matter on this dataset?

# Your code here

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder

feature_cols = ['Customer ID', 'Price', 'Country']   # Quantity excluded — determines y, leakage
y_full = (df['Quantity'] < 0).astype(int)

results = {}

# ── Strategy A: Listwise deletion ──────────────────────────────────────────
# Safe to filter globally before CV: dropping rows based on their own
# missingness pattern uses no statistic learned from the data (no mean,
# no neighbor lookup) — unlike imputation, it can't leak fold information.
df_a = df.dropna(subset=feature_cols)
X_a = df_a[feature_cols]
y_a = (df_a['Quantity'] < 0).astype(int)
print(f"Strategy A: {len(df_a):,} rows remain (dropped {len(df) - len(df_a):,}, "
      f"{(1 - len(df_a)/len(df))*100:.1f}%)")

pipeline_a = Pipeline([
    ('encode', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Country']),
    ], remainder='passthrough')),
    ('model', RandomForestClassifier(n_estimators=200, random_state=420, n_jobs=-1)),
])
results['A: Listwise deletion'] = cross_val_score(pipeline_a, X_a, y_a, cv=5, scoring='roc_auc', n_jobs=-1)

# ── Strategy B: Naive median (no mechanism awareness) ──────────────────────
# Median only ever applies to numeric columns — Country is complete already,
# so it just gets encoded, not imputed.
X_full = df[feature_cols].copy()

preprocessor_b = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), ['Customer ID', 'Price']),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Country']),
])
pipeline_b = Pipeline([
    ('preprocess', preprocessor_b),
    ('model', RandomForestClassifier(n_estimators=200, random_state=420, n_jobs=-1)),
])
results['B: Naive median'] = cross_val_score(pipeline_b, X_full, y_full, cv=5, scoring='roc_auc', n_jobs=-1)

# ── Strategy C: SmartImputer (mechanism-aware) ──────────────────────────────
encode_scale = ColumnTransformer([
    ('num', StandardScaler(), make_column_selector(dtype_include=np.number)),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
     make_column_selector(dtype_include=object)),
])
pipeline_c = Pipeline([
    ('imputer', SmartImputer()),
    ('encode_scale', encode_scale),
    ('model', RandomForestClassifier(n_estimators=200, random_state=420, n_jobs=-1)),
])
results['C: SmartImputer'] = cross_val_score(pipeline_c, X_full, y_full, cv=5, scoring='roc_auc', n_jobs=-1)

# ── Report ──────────────────────────────────────────────────────────────
print(f"\n{'Strategy':<25} {'Mean AUC':>10} {'Std':>10} {'N rows':>10}")
for name, scores in results.items():
    n = len(X_a) if name.startswith('A') else len(X_full)
    print(f"{name:<25} {scores.mean():>10.4f} {scores.std():>10.4f} {n:>10,}")

best = max(results, key=lambda k: results[k].mean())
worst = min(results, key=lambda k: results[k].mean())
gap = results[best].mean() - results[worst].mean()
pooled_std = np.mean([results[best].std(), results[worst].std()])
print(f"\nBest: {best} ({results[best].mean():.4f}) | Worst: {worst} ({results[worst].mean():.4f})")
print(f"Gap: {gap:.4f} vs. typical fold-to-fold std of {pooled_std:.4f}")
print("Gap exceeds fold variance -> difference looks meaningful."
      if gap > pooled_std else
      "Gap is within fold variance -> not clearly meaningful on AUC alone.")

# ── Plot ────────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 5))
plt.boxplot([results[name] for name in results], labels=list(results.keys()))
plt.ylabel('AUC (5-fold CV)')
plt.title('Downstream AUC: Listwise Deletion vs. Naive vs. Mechanism-Aware Imputation')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


Strategy A: 417,534 rows remain (dropped 107,927, 20.5%)


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\user\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\user\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 655, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
  File "c:\Users\user\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 589, in _fit
    X, fitted_transformer = fit_transform_one_cached